# NBA 3.1 HCP features: check the cohort-set files

2026-11-06: NB01-NB06 write every HCP feature table once, split by `COHORT_SET`, so this notebook no longer splits anything. 2026-11-11: the tables are Delta (partitioned by `COHORT_SET`, few large files) instead of Parquet; older Parquet runs are still read correctly by `read_hcp_output`. It only **checks** the 11 tables of a run:

* Folder: `hcp_level_model/dev_pipeline/notebook_output/<run>/hcp_claims/<table>/COHORT_SET=<group>/`
* Groups (from the claims cohort file `cohorts/claims/cohorts.parquet`, built in `00_cohort_config`): `C1-C5_training`, `C6-C10_training`, `C11-C15_training`, `C16-C20_training`, `C21-C24_training`, `C25-C26_validation`, `C27-C28_backtesting`.
* Checks per table: exactly the 7 groups; every group has exactly its cohorts; all `EXPECTED_COHORTS` (28) cohorts; one row per (key, `COHORT_ID`); `DATASET` matches the cohort file.
* Job parameters: `RUN_OUTPUT_DIR` (run to check), `COHORT_SET_TABLES` (comma list, empty = all 11 tables), `EXPECTED_COHORTS` (28; empty = no check).
* Read a table back with `read_hcp_output(path)` or `read_cohort_sets(table, ["C25-C26_validation"])` (00_cohort_config).

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"


In [0]:
%run "../../00_cohort_config"

In [0]:
# 1. Settings: tables of this run's hcp_claims/ folder (written by NB01-NB06)
import os
import pandas as pd

assert HCP_NOTEBOOK_OUTPUT, "RUN_OUTPUT_DIR is not set: set it to the run name to check"
_tables = get_run_param("COHORT_SET_TABLES", None)
TABLES = [t.strip() for t in _tables.split(",")] if _tables else list(HCP_COHORT_TABLES)
assert set(TABLES) <= set(HCP_COHORT_TABLES), f"Unknown tables: {sorted(set(TABLES) - set(HCP_COHORT_TABLES))}"
EXPECTED_SETS = {name: sorted(ids) for name, ids in COHORT_SETS}

print(f"RUN              = {RUN_NAME} | VALIDATION_MODE = {VALIDATION_MODE}")
print(f"FOLDER           = {HCP_NOTEBOOK_OUTPUT}<table>/COHORT_SET=<group>/")
print(f"EXPECTED_COHORTS = {EXPECTED_COHORTS} | tables: {len(TABLES)}")
print("GROUPS           = " + ", ".join(f"{n} ({len(i)})" for n, i in COHORT_SETS))

In [0]:
# 2. Check per table (read back from the files, independent of NB01-NB06):
#    groups = the 7 groups; each group holds exactly its cohorts; all EXPECTED_COHORTS cohorts;
#    one row per (key, COHORT_ID); DATASET = the cohort file's dataset of each cohort.
#    Rows with an empty key (NPIs without PROVIDER_ID, open point O2) are counted in NULL_KEYS and left out of
#    the duplicate check (a distinct count would merge them into one key per cohort).
#    Groups and their cohorts are read from the table (read_hcp_output: Delta from 2026-11-11, Parquet for older
#    runs), not from the folders: a Delta folder can hold files that are no longer part of the table.
checks = []
for t in TABLES:
    key, path = HCP_COHORT_TABLES[t], HCP_NOTEBOOK_OUTPUT + t + "/"
    if not os.path.exists(path):
        checks.append({"TABLE": t, "OK": False, "PROBLEM": f"missing: {path}"}); continue
    df = read_hcp_output(path)
    per_set = {}
    for r in df.select("COHORT_SET", "COHORT_ID").distinct().collect():
        per_set.setdefault(r[0], []).append(r[1])
    per_set = {n: sorted(ids) for n, ids in per_set.items()}
    folders = sorted(per_set)
    n_rows = df.count()
    n_null = df.where(F.col(key).isNull()).count()   # NPIs without PROVIDER_ID (O2), reported, not a duplicate
    n_keys = df.where(F.col(key).isNotNull()).select(F.col(key).cast("string"), "COHORT_ID").distinct().count()
    _ds = {r[0]: r[1] for r in df.select("COHORT_ID", "DATASET").distinct().collect()}
    problems = []
    if set(folders) != set(EXPECTED_SETS):
        problems.append(f"groups {folders}")
    problems += [f"{n}: {per_set.get(n)} != {ids}" for n, ids in EXPECTED_SETS.items()
                 if n in per_set and sorted(per_set[n], key=lambda c: int(c[1:])) != sorted(ids, key=lambda c: int(c[1:]))]
    n_coh = len({c for ids in per_set.values() for c in ids})
    if EXPECTED_COHORTS and n_coh != EXPECTED_COHORTS:
        problems.append(f"{n_coh} cohorts, expected {EXPECTED_COHORTS}")
    if n_rows - n_null != n_keys:
        problems.append(f"{n_rows - n_null - n_keys} duplicate ({key}, COHORT_ID)")
    problems += [f"{c}: DATASET {d} != {COHORT_DATASET.get(c)}" for c, d in _ds.items() if d != COHORT_DATASET.get(c)]
    checks.append({"TABLE": t, "OK": not problems, "GROUPS": len(folders), "COHORTS": n_coh, "ROWS": n_rows,
                   "KEYS": n_keys, "NULL_KEYS": n_null, "PROBLEM": "; ".join(problems) or None})
checks = pd.DataFrame(checks)
display(checks)
assert checks.OK.all(), f"Cohort-set check failed for: {checks[~checks.OK].TABLE.tolist()}"
print(f"All {len(checks)} tables OK: {len(EXPECTED_SETS)} groups, {EXPECTED_COHORTS} cohorts, one row per key and cohort")